In [1]:
import os, json, random, collections
import pandas as pd
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

random.seed(42)
PLACES = '/kaggle/input/datasets/benjaminkz/places365/train'
BDD = '/kaggle/input/datasets/solesensei/solesensei_bdd100k'
LAB = BDD + '/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_%s.json'
IMG = BDD + '/bdd100k/bdd100k/images/100k/%s'
OUT = '/kaggle/working/data'
CAP, MIN_AREA, MIN_CLASS = 1500, 3000, 200

def save(src, dst, size):
    try:
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        Image.open(src).convert('RGB').resize(size).save(dst, quality=92)
        return True
    except Exception as e:
        print('skip', src, e)
        return False

# ---------- Head 1: Places365 (48 classes, 16 per super-group) ----------
chosen = ['bedroom','kitchen','bathroom','living_room','classroom','library-indoor','restaurant','supermarket','church-indoor','gymnasium-indoor','hospital_room','airplane_cabin','bookstore','bar','corridor','museum-indoor',
 'beach','forest-broadleaf','mountain','desert-sand','waterfall','canyon','rice_paddy','glacier','lake-natural','volcano','rainforest','wheat_field','swamp','tundra','river','badlands',
 'highway','street','skyscraper','bridge','castle','lighthouse','stadium-football','gas_station','parking_lot','railroad_track','windmill','harbor','construction_site','amusement_park','cemetery','playground']
rows = []
for gi, c in enumerate(chosen):
    fs = sorted(os.listdir(f'{PLACES}/{c}'))
    for i, f in enumerate(random.sample(fs, 500)):
        sp = 'train' if i < 400 else 'val' if i < 450 else 'test'
        rows.append((f'{PLACES}/{c}/{f}', f'places/{sp}/{c}/{f}', c, gi // 16, sp))
pl = pd.DataFrame(rows, columns=['src', 'path', 'cls', 'group', 'split'])
with ThreadPoolExecutor(16) as ex:
    ok = list(ex.map(lambda r: save(r[0], f'{OUT}/{r[1]}', (224, 224)), rows))
pl = pl[ok]
pl['label'] = pl.cls.map({c: i for i, c in enumerate(chosen)})
pl.drop(columns='src').to_csv('/kaggle/working/places_manifest.csv', index=False)
print('PLACES', len(pl)); print(pl.split.value_counts())

# ---------- Heads 2+3: BDD100K ----------
W = {'clear': 'clear', 'partly cloudy': 'clear', 'overcast': 'overcast', 'rainy': 'rainy', 'snowy': 'snow_fog', 'foggy': 'snow_fog'}
T = {'daytime': 'day', 'night': 'night', 'dawn/dusk': 'twilight'}
pool = []
for sp in ['train', 'val']:
    for x in json.load(open(LAB % sp)):
        a = x['attributes']
        if a['weather'] not in W or a['timeofday'] not in T:
            continue
        ps = [(b['box2d']['x2'] - b['box2d']['x1']) * (b['box2d']['y2'] - b['box2d']['y1'])
              for b in x['labels'] if b['category'] in ('person', 'rider') and 'box2d' in b]
        if ps and max(ps) < MIN_AREA:   # only tiny people: ambiguous, skip
            continue
        pool.append((IMG % sp + '/' + x['name'], W[a['weather']] + '_' + T[a['timeofday']], int(bool(ps))))
by = collections.defaultdict(list)
for p in pool:
    by[p[1]].append(p)
print({k: len(v) for k, v in sorted(by.items())})
classes = sorted(k for k, v in by.items() if len(v) >= MIN_CLASS)
sel = []
for k in classes:
    v = by[k]; random.shuffle(v); n = min(len(v), CAP)
    for i, p in enumerate(v[:n]):
        sp = 'train' if i < 0.83 * n else 'val' if i < 0.915 * n else 'test'
        sel.append((p[0], f'bdd/{sp}/{os.path.basename(p[0])}', k, classes.index(k), p[2], sp))
bd = pd.DataFrame(sel, columns=['src', 'path', 'cond', 'cond_id', 'person', 'split'])
with ThreadPoolExecutor(16) as ex:
    ok = list(ex.map(lambda r: save(r[0], f'{OUT}/{r[1]}', (384, 216)), sel))
bd = bd[ok]
bd.drop(columns='src').to_csv('/kaggle/working/bdd_manifest.csv', index=False)
json.dump({'places': chosen, 'cond': classes}, open('/kaggle/working/class_names.json', 'w'))
print('BDD', len(bd)); print(bd.split.value_counts())
print(bd.groupby('cond').size()); print('person rate', bd.person.mean())

PLACES 24000
split
train    19200
val       2400
test      2400
Name: count, dtype: int64
{'clear_day': 14829, 'clear_night': 23089, 'clear_twilight': 2467, 'overcast_day': 6495, 'overcast_night': 78, 'overcast_twilight': 1043, 'rainy_day': 2354, 'rainy_night': 2196, 'rainy_twilight': 322, 'snow_fog_day': 2558, 'snow_fog_night': 2240, 'snow_fog_twilight': 422}
skip /kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train/8346c421-fa681916.jpg [Errno 2] No such file or directory: '/kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train/8346c421-fa681916.jpg'
skip /kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train/7845b80e-12c3c18a.jpg [Errno 2] No such file or directory: '/kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train/7845b80e-12c3c18a.jpg'
skip /kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train/6b4fdcf1-3111d912.jpg